In [38]:
import numpy as np

In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer


In [1]:
from sklearn.preprocessing import StandardScaler

In [41]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [42]:
data=load_breast_cancer()
X=data.data
Y=data.target


In [43]:
#split the dataset into training and test set
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)


In [44]:
#standardize the data using standard scalar

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [45]:
type(X_train)

numpy.ndarray

In [46]:
#convert the data to pytorch tensor and move it to gpu
X_train=torch.tensor(X_train,dtype=torch.float32).to(device)
Y_train=torch.tensor(Y_train,dtype=torch.float32).to(device)
X_test=torch.tensor(X_test,dtype=torch.float32).to(device)
Y_test=torch.tensor(Y_test,dtype=torch.float32).to(device)

In [47]:
#define the neural network architecture
class NeuralNet(nn.Module):
    def __init__(self, input_size,hidden_size,output_size):
        super(NeuralNet,self).__init__()
        self.fc1=nn.Linear(input_size,hidden_size)
        self.relu=nn.ReLU()
        self.fc2=nn.Linear(hidden_size,output_size)
        self.sigmoid=nn.Sigmoid()
        
    def forward(self,x):
        out=self.fc1(x)
        out=self.relu(out)
        out=self.fc2(out)
        out=self.sigmoid(out)
        return out

In [48]:
#define hyperparameters
input_size=X_train.shape[1]
hidden_size=64
output_size=1
learning_rate=0.001
num_epochs=100

In [49]:
#initialize the neural network and move it to gpu
model=NeuralNet(input_size,hidden_size,output_size).to(device)

In [50]:
#define loss and optimizer
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate)


In [51]:
#training the neural network
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs=model(X_train)
    loss=criterion(outputs,Y_train.view(-1,1))
    loss.backward()
    optimizer.step()
    
    # calculate accuracy
    with torch.no_grad():
        predicted=outputs.round()
        correct=(predicted==Y_train.view(-1,1)).float().sum()
        accuracy=correct/Y_train.size(0)
        
    if (epoch+1)%10==0:
        print(f"Epoch: [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy.item()*100:.2f}%") 
           
        

Epoch: [10/100], Loss: 0.5326, Accuracy: 89.45%
Epoch: [20/100], Loss: 0.4175, Accuracy: 91.65%
Epoch: [30/100], Loss: 0.3272, Accuracy: 93.41%
Epoch: [40/100], Loss: 0.2601, Accuracy: 94.73%
Epoch: [50/100], Loss: 0.2121, Accuracy: 95.38%
Epoch: [60/100], Loss: 0.1782, Accuracy: 95.60%
Epoch: [70/100], Loss: 0.1537, Accuracy: 96.70%
Epoch: [80/100], Loss: 0.1353, Accuracy: 96.70%
Epoch: [90/100], Loss: 0.1210, Accuracy: 97.36%
Epoch: [100/100], Loss: 0.1096, Accuracy: 97.80%


In [52]:
#evaluation on training set
model.eval()
with torch.no_grad():
    outputs=model(X_train)
    predicted=outputs.round()
    correct=(predicted==Y_train.view(-1,1)).float().sum()
    accuracy=correct/Y_train.size(0)
    print(f"Accuracy on training data: {accuracy.item()*100:.2f}%")


Accuracy on training data: 98.02%


In [53]:
model.eval()
with torch.no_grad():
    outputs=model(X_test)
    predicted=outputs.round()
    correct=(predicted==Y_test.view(-1,1)).float().sum()
    accuracy=correct/Y_test.size(0)
    print(f"Accuracy on testing data: {accuracy.item()*100:.2f}%")

Accuracy on testing data: 98.25%
